# Código de repetición y corrección de errores

Este cuaderno muestra un código corrector muy simple: repetir cada bit varias veces y decodificar por mayoría.

No es un código eficiente, pero permite ver la idea básica de añadir redundancia para combatir ruido.

In [ ]:
import random


def codificar_repeticion(bits, repeticiones=3):
    return [bit for bit in bits for _ in range(repeticiones)]


def canal_binario(bits, p_error, semilla=0):
    rng = random.Random(semilla)
    return [1 - bit if rng.random() < p_error else bit for bit in bits]


def decodificar_mayoria(bits, repeticiones=3):
    if len(bits) % repeticiones != 0:
        raise ValueError("La longitud no es múltiplo del número de repeticiones.")
    salida = []
    for i in range(0, len(bits), repeticiones):
        bloque = bits[i:i + repeticiones]
        salida.append(1 if sum(bloque) > repeticiones / 2 else 0)
    return salida


def tasa_error(original, recibido):
    return sum(a != b for a, b in zip(original, recibido)) / len(original)

## Un mensaje pequeño

Codificamos, transmitimos con ruido y decodificamos por mayoría.

In [ ]:
mensaje = [1, 0, 1, 1, 0, 0, 1, 0]
codificado = codificar_repeticion(mensaje, repeticiones=3)
recibido = canal_binario(codificado, p_error=0.12, semilla=4)
decodificado = decodificar_mayoria(recibido, repeticiones=3)

print("mensaje:     ", mensaje)
print("codificado:  ", codificado)
print("recibido:    ", recibido)
print("decodificado:", decodificado)
print("error final: ", tasa_error(mensaje, decodificado))

## Comparación experimental

Comparamos enviar bits sin protección frente a repetir cada bit tres veces.

In [ ]:
rng = random.Random(17)
mensaje_largo = [rng.randrange(2) for _ in range(5000)]

for p in [0.01, 0.05, 0.1, 0.2]:
    sin_proteccion = canal_binario(mensaje_largo, p, semilla=21)
    codificado = codificar_repeticion(mensaje_largo, 3)
    recibido = canal_binario(codificado, p, semilla=21)
    corregido = decodificar_mayoria(recibido, 3)
    print(f"p = {p:.2%}  sin protección = {tasa_error(mensaje_largo, sin_proteccion):.2%}  repetición = {tasa_error(mensaje_largo, corregido):.2%}")

## Distancia de Hamming

La distancia entre las palabras `000` y `111` es 3. Por eso el código puede corregir un error por bloque.

In [ ]:
def hamming(a, b):
    if len(a) != len(b):
        raise ValueError("Las cadenas deben tener la misma longitud.")
    return sum(x != y for x, y in zip(a, b))


print("d(000, 111) =", hamming("000", "111"))
print("d(000, 001) =", hamming("000", "001"))
print("d(111, 101) =", hamming("111", "101"))

## Para experimentar

1. Cambia `repeticiones` a 5 y compara resultados.
2. Sube `p_error` hasta que la repetición deje de ayudar claramente.
3. Calcula la tasa del código para 3, 5 y 7 repeticiones.